In [24]:
pip install TA-Lib

   ---------------------------------------- 0.0/920.2 kB ? eta -:--:--
   ---------------------------------- ----- 786.4/920.2 kB 8.5 MB/s eta 0:00:01
   ---------------------------------------- 920.2/920.2 kB 8.4 MB/s  0:00:00

   ------------- -------------------------- 1/3 [build]
   ------------- -------------------------- 1/3 [build]
   ------------- -------------------------- 1/3 [build]
   -------------------------- ------------- 2/3 [TA-Lib]
   ---------------------------------------- 3/3 [TA-Lib]

Note: you may need to restart the kernel to use updated packages.


In [25]:
import yfinance as yf
import matplotlib.pyplot as plt
import pandas as pd
from datetime import datetime
import talib

In [8]:
stock_codes = ["ABEV3", "ASAI3", "AXIA3"]
stock_codes_with_sa = [code + '.SA' for code in stock_codes]
print(stock_codes_with_sa)

['ABEV3.SA', 'ASAI3.SA', 'AXIA3.SA']


In [9]:
yf.download(stock_codes_with_sa, start='2024-01-01', end='2025-01-01') 

[*********************100%***********************]  3 of 3 completed


Price           Close                             High                        \
Ticker       ABEV3.SA   ASAI3.SA   AXIA3.SA   ABEV3.SA   ASAI3.SA   AXIA3.SA   
Date                                                                           
2024-01-02  12.088682  12.618600  36.742970  12.106317  13.035343  37.297545   
2024-01-03  11.991691  12.541066  36.698952  12.123952  12.715517  37.103884   
2024-01-04  12.035777  12.802741  36.382053  12.062230  12.919041  36.839797   
2024-01-05  11.991691  13.161335  36.320427  12.071047  13.616845  36.681342   
2024-01-08  12.088682  13.733146  36.144379  12.123952  13.733146  36.399658   
...               ...        ...        ...        ...        ...        ...   
2024-12-20  11.378070   5.592114  31.348095  11.443194   5.659956  31.623938   
2024-12-23  11.368767   5.611497  30.920990  11.480408   5.718106  31.303610   
2024-12-26  11.294340   5.727798  31.107847  11.387374   5.747181  31.410384   
2024-12-27  11.229216   5.553347  31.143440  11.340856   5.776256  31.463773   
2024-12-30  10.922204   5.456430  31.137032  11.312947   5.640572  31.237386   

Price             Low                             Open                        \
Ticker       ABEV3.SA   ASAI3.SA   AXIA3.SA   ABEV3.SA   ASAI3.SA   AXIA3.SA   
Date                                                                           
2024-01-02  11.982873  12.502299  36.540502  12.097500  13.035343  36.989447   
2024-01-03  11.965238  12.318157  36.575713  12.053412  12.628292  36.795783   
2024-01-04  11.921151  12.444148  36.047545  11.991691  12.502298  36.751770   
2024-01-05  11.929968  12.744591  36.100356  12.009325  12.802742  36.267612   
2024-01-08  11.974056  13.093493  35.827477  12.000508  13.151643  36.294025   
...               ...        ...        ...        ...        ...        ...   
2024-12-20  11.173395   5.165679  31.063355  11.238519   5.223829  31.534955   
2024-12-23  11.154788   5.475813  30.814213  11.266429   5.514580  31.232423   
2024-12-26  11.154788   5.504888  30.876497  11.331554   5.601806  30.965477   
2024-12-27  11.136182   5.437046  31.072254  11.312946   5.766564  31.241320   
2024-12-30  10.922204   5.330438  30.653507  11.219914   5.582422  30.936323   

Price         Volume                      
Ticker      ABEV3.SA  ASAI3.SA  AXIA3.SA  
Date                                      
2024-01-02  11690200   7348700   3568400  
2024-01-03  17612200   6830500   3762800  
2024-01-04  19996000   8604100   3017000  
2024-01-05  18293900  12696400   3085000  
2024-01-08  11014600  17538900   3643100  
...              ...       ...       ...  
2024-12-20  86074600  45865200  16048900  
2024-12-23  40181700  24589900   6615800  
2024-12-26  21166800  25976300   5387300  
2024-12-27  37624000  30175900   7961400  
2024-12-30  38907400  17975800   6821300  

[251 rows x 15 columns]

In [35]:
def obter_cotacoes(data_referencia, codigos_acoes):
    lista_dfs = []

    for cdgo in codigos_acoes:
        df = yf.download(cdgo, start=data_referencia, progress=False)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.droplevel(1)
            
        df = df.reset_index()
        df['CD_ACAO'] = cdgo
        
        # Garante que as colunas existem antes de selecionar
        if 'Close' in df.columns and 'Volume' in df.columns:
            df = df[['Date', 'CD_ACAO', 'Close', 'Volume']]
            lista_dfs.append(df)
    
    if lista_dfs:
        df_final = pd.concat(lista_dfs, ignore_index=True)
        return df_final
    else:
        return pd.DataFrame()
data_referencia = '2025-01-01'

In [36]:
ne = obter_cotacoes(data_referencia, stock_codes_with_sa)

In [12]:
def calcular_rsi(df, periodos=[6, 14, 20], coluna_preco='Close', coluna_acao='CD_ACAO'):
    """
    Calcula o RSI (Wilder) para um DataFrame. 
    Se houver a coluna 'CD_ACAO', calcula separadamente para cada ativo.
    """
    
    # Função interna que faz o cálculo matemático para uma única ação
    def _rsi_core(df_grupo):
        df_grupo = df_grupo.sort_values('Date') if 'Date' in df_grupo.columns else df_grupo
        
        delta = df_grupo[coluna_preco].diff()
        
        # Separa ganhos e perdas
        ganhos = delta.where(delta > 0, 0)
        perdas = -delta.where(delta < 0, 0)
        
        for n in periodos:
            # Wilder's Smoothing (MME com alpha=1/n)
            media_ganhos = ganhos.ewm(alpha=1/n, min_periods=n, adjust=False).mean()
            media_perdas = perdas.ewm(alpha=1/n, min_periods=n, adjust=False).mean()
            
            rs = media_ganhos / media_perdas
            rsi = 100 - (100 / (1 + rs))
            
            # Tratamento para divisão por zero (caso raro)
            df_grupo[f'RSI_{n}'] = rsi.fillna(100)
            
        return df_grupo

    # --- Lógica Principal ---
    # Verifica se existe a coluna de código da ação para agrupar
    if coluna_acao in df.columns:
        # Aplica a função _rsi_core para cada grupo de ação
        df_resultado = df.groupby(coluna_acao, group_keys=False).apply(_rsi_core)
    else:
        # Se não tiver coluna de ação, assume que é uma ação só
        df_resultado = _rsi_core(df.copy())
        
    return df_resultado

In [ ]:
RSI = calcular_rsi(ne)


C:\Users\rodri\AppData\Local\Temp\ipykernel_2328\3681598418.py:34: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_resultado = df.groupby(coluna_acao, group_keys=False).apply(_rsi_core)


In [ ]:
ticker = yf.Ticker("ASAI3.SA")

AttributeError: 'DataFrame' object has no attribute 'ta'

In [39]:
rsi2= talib.RSI(ne['Close'].values)
macd2 = talib.OBV(ne['Close'].values,ne['Volume'].astype(float).values)